# N-Body Gravitational Simulation

The gravitational N-body problem computes pairwise forces between every pair of particles
and integrates their trajectories over time. This notebook uses the Minkowski ECS Python
bridge to simulate 200 bodies under mutual gravitational attraction.

The approach here is **O(N-squared) pairwise gravity** — every body exerts a force on every other
body each frame. (The Rust `nbody` example uses a Barnes-Hut quadtree for O(N log N), but
the Python bridge reducer keeps things simpler with direct summation.)

In [ ]:
import random

import matplotlib.pyplot as plt
import minkowski_py as mk
import numpy as np
import polars as pl
from IPython.display import clear_output

# Constants
N_BODIES = 200
WORLD_SIZE = 500.0
FRAMES = 500
DT = 0.005
G = 0.5
SOFTENING = 2.0

## Spawn Bodies

Two clusters of 100 bodies each, offset from center. A handful of heavy bodies (mass 5--20)
act as gravitational attractors; the rest have mass 1.0.

In [ ]:
random.seed(42)

world = mk.World()

masses = []
pos_x = []
pos_y = []
vel_x = []
vel_y = []

# Cluster A: centered at (150, 250)
for i in range(100):
    px = 150.0 + random.gauss(0, 30)
    py = 250.0 + random.gauss(0, 30)
    vx = random.gauss(0, 2)
    vy = random.gauss(0, 2)
    # First 5 bodies in each cluster are heavy attractors
    m = random.uniform(5.0, 20.0) if i < 5 else 1.0
    masses.append(m)
    pos_x.append(px)
    pos_y.append(py)
    vel_x.append(vx)
    vel_y.append(vy)

# Cluster B: centered at (350, 250)
for i in range(100):
    px = 350.0 + random.gauss(0, 30)
    py = 250.0 + random.gauss(0, 30)
    vx = random.gauss(0, 2)
    vy = random.gauss(0, 2)
    m = random.uniform(5.0, 20.0) if i < 5 else 1.0
    masses.append(m)
    pos_x.append(px)
    pos_y.append(py)
    vel_x.append(vx)
    vel_y.append(vy)

world.spawn_batch(
    "Mass,Position,Velocity",
    {
        "mass": masses,
        "pos_x": pos_x,
        "pos_y": pos_y,
        "vel_x": vel_x,
        "vel_y": vel_y,
    },
)

print(f"Spawned {N_BODIES} bodies")

In [ ]:
registry = mk.ReducerRegistry(world)

## Run Simulation

Each frame calls the `gravity` reducer which computes O(N-squared) pairwise gravitational forces,
updates velocities, and integrates positions. We store snapshots every 10 frames for animation.

In [ ]:
snapshots = []

for frame in range(FRAMES):
    registry.run("gravity", world, g=G, softening=SOFTENING, dt=DT)

    if frame % 10 == 0:
        df = world.query("Position", "Velocity", "Mass")
        snapshots.append((frame, df))

    if frame % 100 == 0:
        print(f"Frame {frame}/{FRAMES}")

print(f"Done. Collected {len(snapshots)} snapshots.")

## Visualize Final State

Scatter plot of body positions at the last frame. Point size is proportional to mass,
color encodes velocity magnitude.

In [ ]:
final_frame, final_df = snapshots[-1]

x = final_df["pos_x"].to_numpy()
y = final_df["pos_y"].to_numpy()
vx = final_df["vel_x"].to_numpy()
vy = final_df["vel_y"].to_numpy()
masses = final_df["mass"].to_numpy()
speed = np.sqrt(vx**2 + vy**2)

fig, ax = plt.subplots(figsize=(8, 8))
sc = ax.scatter(
    x, y, s=masses * 10, c=speed, cmap="inferno", alpha=0.8, edgecolors="none"
)
ax.set_xlim(0, WORLD_SIZE)
ax.set_ylim(0, WORLD_SIZE)
ax.set_aspect("equal")
ax.set_title(f"N-Body — Frame {final_frame}")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.colorbar(sc, ax=ax, label="Speed")
plt.tight_layout()
plt.show()

## Animation

Loop through stored snapshots with inline animation. Point size scales with mass,
color encodes speed.

In [ ]:
for _frame_idx, (frame, df) in enumerate(snapshots):
    x = df["pos_x"].to_numpy()
    y = df["pos_y"].to_numpy()
    vx = df["vel_x"].to_numpy()
    vy = df["vel_y"].to_numpy()
    m = df["mass"].to_numpy()
    speed = np.sqrt(vx**2 + vy**2)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(x, y, s=m * 10, c=speed, cmap="inferno", alpha=0.8, edgecolors="none")
    ax.set_xlim(0, WORLD_SIZE)
    ax.set_ylim(0, WORLD_SIZE)
    ax.set_aspect("equal")
    ax.set_title(f"N-Body — Frame {frame}")
    ax.set_facecolor("black")
    fig.patch.set_facecolor("black")
    ax.tick_params(colors="white")
    ax.xaxis.label.set_color("white")
    ax.yaxis.label.set_color("white")
    ax.title.set_color("white")

    clear_output(wait=True)
    plt.show()

print("Animation complete.")

## Energy Analysis

Compute total kinetic energy from the final state using Polars expressions.
Without pairwise potential energy tracking, this is KE only — a full energy audit
would require summing gravitational potential over all pairs.

In [ ]:
ke_df = final_df.select(
    (0.5 * pl.col("mass") * (pl.col("vel_x") ** 2 + pl.col("vel_y") ** 2)).alias("ke")
)

total_ke = ke_df["ke"].sum()
print(f"Total kinetic energy at frame {final_frame}: {total_ke:.4f}")

## Explore

Things to try:

- **Stronger gravity**: increase `G` (e.g. 2.0) to see faster cluster merging
- **Harder collisions**: decrease `SOFTENING` (e.g. 0.5) for tighter interactions — but small values can cause numerical instability with large `DT`
- **Central star**: set one body to mass 500+ at the center, give the rest small circular velocities for an orbit simulation
- **Performance**: if frames are slow, reduce `N_BODIES` — the cost is O(N-squared) per frame